<a href="https://colab.research.google.com/github/vutting895/VOOM_SYSTEM-NOVA-AI/blob/main/colab/nova-auto-learning/NOVA_Colab_Runtime_Preflight.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NOVA Google Colab Runtime Preflight

**Purpose:** verify the interactive Colab runtime and optional Gemini credential presence before any NOVA research work. This notebook does **not** download market data, train a model, upload artifacts, call a broker, or deploy services.

Keep this notebook private. Never paste secrets into cells or outputs. Do not mount Google Drive unless a later reviewed step explicitly requires it.


In [ ]:
import importlib.util, os, platform, sys
from pathlib import Path

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('CPU count:', os.cpu_count())
print('Colab runtime:', importlib.util.find_spec('google.colab') is not None)

for package in ('numpy', 'pandas', 'sklearn', 'xgboost', 'torch', 'requests'):
    print(f'{package}:', 'available' if importlib.util.find_spec(package) else 'not installed')

try:
    import psutil
    print('RAM GiB:', round(psutil.virtual_memory().total / (1024**3), 2))
except Exception:
    print('RAM GiB: unavailable (psutil not installed)')


In [ ]:
def secret_is_configured(name):
    value = os.environ.get(name, '').strip()
    if value:
        return True
    try:
        from google.colab import userdata
        return bool((userdata.get(name) or '').strip())
    except Exception:
        return False

print('GEMINI_API_KEY configured:', secret_is_configured('GEMINI_API_KEY'))
print('NOVA_MT5_BRIDGE_URL configured:', secret_is_configured('NOVA_MT5_BRIDGE_URL'))
print('NOVA_MT5_BRIDGE_TOKEN configured:', secret_is_configured('NOVA_MT5_BRIDGE_TOKEN'))
print('NOVA_MT5_TENANT_ID configured:', secret_is_configured('NOVA_MT5_TENANT_ID'))
print('Secret values were not displayed or sent anywhere.')


In [ ]:
# Fail-closed research gate. This preflight notebook intentionally contains no model.fit() calls.
TRAINING_ENABLED = False
DATA_QUALITY_GATE_STATUS = 'NOT_RUN'  # Must be set by a separately reviewed validator, never by assumption.
EXPLICIT_RESEARCH_APPROVAL = False

if TRAINING_ENABLED and DATA_QUALITY_GATE_STATUS != 'PASS':
    raise RuntimeError('Training blocked: the verified dataset quality gate has not passed.')
if TRAINING_ENABLED and not EXPLICIT_RESEARCH_APPROVAL:
    raise RuntimeError('Training blocked: explicit research approval is required.')

print('Training started: False')
print('Reason: this notebook is preflight-only; dataset eligibility and training code are not activated.')
print('NOVA trading safety: READ-ONLY; ordersEnabled=False; executionEnabled=False; liveTradingEnabled=False')


## Next step

Run all cells interactively in Google Colab and record the output. A successful runtime preflight proves only that the notebook environment is available; it does not prove that the Exness historical dataset is complete, that the Gemini API quota is available, or that a model has trained successfully.

The current Exness XAUUSDm H1 dataset remains blocked by a 7,419-hour missing segment. Do not start model training until a separately verified historical source closes that gap and the repository quality gate passes.

Google Colab resources are not guaranteed; availability and usage limits vary. See the [official Colab FAQ](https://research.google.com/colaboratory/faq.html).


### 1. Model Audit & Environment Discovery
ตามคู่มือควบคุมความปลอดภัยของ NOVA ก่อนดำเนินงานอื่นใด เราจำเป็นต้องสแกนหาโมเดลเดิมและตรวจสอบความพร้อมของพื้นที่จัดเก็บข้อมูลใน Google Drive `/content/drive/MyDrive/NOVA/` เพื่อรายงานสถานะให้ครบถ้วน

In [1]:
import os
from pathlib import Path

# กำหนดตำแหน่งที่ตั้งหลักตามที่ระบุในคู่มือระบบ NOVA
NOVA_BASE_DIR = Path('/content/drive/MyDrive/NOVA')

sub_directories = [
    'approved-market-data',
    'approved-oof',
    'checkpoints',
    'trained-models',
    'evaluation'
]

def audit_nova_workspace():
    print("=== NOVA Workspace & Model Audit ===")

    # 1. ตรวจสอบการ Mount Google Drive
    gdrive_mounted = os.path.exists('/content/drive')
    print(f"Google Drive Mounted: {gdrive_mounted}")

    if not gdrive_mounted:
        print("[!] Google Drive ยังไม่ได้เชื่อมต่อ หากต้องการตรวจสอบไฟล์ใน Drive กรุณา Run Cell สำหรับ Mount Drive ก่อน")
        return

    # 2. ตรวจสอบ Directory สำคัญ
    print(f"\nChecking base directory: {NOVA_BASE_DIR}")
    if NOVA_BASE_DIR.exists():
        print("[PASS] Base directory exists.")
        for sub in sub_directories:
            path = NOVA_BASE_DIR / sub
            status = "[EXISTS]" if path.exists() else "[MISSING]"
            print(f"  - {sub}/: {status}")
    else:
        print("[WARNING] NOVA Base directory not found. Please create it or check your Google Drive path.")

    # 3. สแกนหาโมเดลเดิม (เช่น .onnx, .pt, .pkl)
    print("\nScanning for existing models...")
    model_extensions = ('*.onnx', '*.pt', '*.pth', '*.pkl', '*.h5')
    found_models = []

    if NOVA_BASE_DIR.exists():
        for ext in model_extensions:
            found_models.extend(list(NOVA_BASE_DIR.glob(f"**/{ext}")))

    if found_models:
        print(f"Found {len(found_models)} model file(s):")
        for m in found_models:
            print(f"  - {m.relative_to(NOVA_BASE_DIR)}")
    else:
        print("No existing model files found in /content/drive/MyDrive/NOVA/ [Status: NOT FOUND]")

audit_nova_workspace()

=== NOVA Workspace & Model Audit ===
Google Drive Mounted: False
[!] Google Drive ยังไม่ได้เชื่อมต่อ หากต้องการตรวจสอบไฟล์ใน Drive กรุณา Run Cell สำหรับ Mount Drive ก่อน


### 2. Initial Audit Status Report
ตารางรายงานสถานะเบื้องต้นตามข้อกำหนดของระบบ NOVA

In [2]:
import pandas as pd

audit_data = {
    "Item": [
        "1. Google Drive Connection",
        "2. NOVA Base Directory Check",
        "3. Existing Model Discovery",
        "4. Dataset Validation Check",
        "5. Target Label & Feature Schema"
    ],
    "Status": [
        "NOT VERIFIED",
        "NOT VERIFIED",
        "NOT FOUND",
        "BLOCKED (7,419-hour gap reported)",
        "NOT VERIFIED"
    ],
    "Details": [
        "Requires standard Mount Drive",
        "Checks path /content/drive/MyDrive/NOVA",
        "No active ONNX/PyTorch models detected yet",
        "Exness XAUUSDm H1 dataset requires fix",
        "Pending dataset ingestion validation"
    ]
}

df_report = pd.DataFrame(audit_data)
display(df_report)

,Item,Status,Details
0,1. Google Drive Connection,NOT VERIFIED,Requires standard Mount Drive
1,2. NOVA Base Directory Check,NOT VERIFIED,Checks path /content/drive/MyDrive/NOVA
2,3. Existing Model Discovery,NOT FOUND,No active ONNX/PyTorch models detected yet
3,4. Dataset Validation Check,"BLOCKED (7,419-hour gap reported)",Exness XAUUSDm H1 dataset requires fix
4,5. Target Label & Feature Schema,NOT VERIFIED,Pending dataset ingestion validation
